# Session figures — emotion + speaker (guaranteed consistent)
Draws **both** figures in one run from the **same** session pick and segment order, so they line up bar-for-bar:
- `session_figure.pdf` — colour = emotion label (green PZ / red NG / grey excluded)
- `speaker_figure.pdf` — colour = speaker (blue child / orange therapist / grey excluded)

Reads your saved `labels.csv`; nothing is retrained. Edit `LABELS_CSV` if needed, then Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import Image, display

LABELS_CSV = "/content/drive/MyDrive/CHANGE_ME/labeling_outputs/labels.csv"
OUT_DIR    = os.path.dirname(LABELS_CSV)
N_SESSIONS = 2
FIGSIZE    = lambda n: (7.6, 0.8 * n + 0.6)   # box height matches across both figures
LAB_COL = {"PZ": "#2e7d32", "NG": "#c62828", "EXCLUDE": "#dcdcdc"}
SPK_COL = {"child": "#1f77b4", "therapist": "#ff7f0e"}
GREY    = "#dcdcdc"

# ---- load once ----
df = pd.read_csv(LABELS_CSV, dtype={"id": str, "session": str})
df = df.dropna(subset=["session", "seg_idx"])
df["seg_idx"] = df["seg_idx"].astype(int)
df = df.sort_values(["session", "seg_idx"]).reset_index(drop=True)
df["labeled"] = df["outcome"] == "labeled"
df["lab"]     = df["final_label"].where(df["labeled"], "EXCLUDE")
df["role"]    = df["speaker_role"].astype(str).str.lower()

# ---- pick sessions ONCE (deterministic), used by BOTH figures ----
cnt = (df[df["labeled"]].groupby("session").size().rename("labeled").reset_index()
       .sort_values(["labeled", "session"], ascending=[False, True]))
picks = cnt["session"].head(N_SESSIONS).tolist()
cnt.to_csv(os.path.join(OUT_DIR, "session_inventory.csv"), index=False)
print("sessions:", [s[:45] for s in picks])

def draw(kind, out):
    fig, axes = plt.subplots(len(picks), 1, figsize=FIGSIZE(len(picks)))
    if len(picks) == 1: axes = [axes]
    for k, (ax, s) in enumerate(zip(axes, picks)):
        g = df[df["session"] == s].sort_values("seg_idx").reset_index(drop=True)
        for i, r in g.iterrows():
            if kind == "emotion":
                col = LAB_COL.get(r["lab"], GREY)
            else:  # speaker
                col = SPK_COL.get(r["role"], GREY) if r["labeled"] else GREY
            ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=col, linewidth=0))
        ax.set_xlim(0, len(g)); ax.set_ylim(0, 1); ax.set_yticks([])
        ax.set_xlabel("segment order within session", fontsize=8)
        nlab = int(g["labeled"].sum())
        if kind == "emotion":
            npz = int((g["lab"] == "PZ").sum()); nng = int((g["lab"] == "NG").sum())
            sub = f"{nlab} labeled ({npz} PZ / {nng} NG)"
        else:
            nc = int(((g["role"] == "child") & g["labeled"]).sum())
            nt = int(((g["role"] == "therapist") & g["labeled"]).sum())
            sub = f"{nc} child / {nt} therapist (labeled turns)"
        ax.set_title(f"Session {chr(65 + k)}   —   {len(g)} segments · {sub}",
                     fontsize=8.5, loc="left", pad=3)
        ax.tick_params(labelsize=7)
    if kind == "emotion":
        handles = [Patch(facecolor=LAB_COL["PZ"], label="Positive (PZ)"),
                   Patch(facecolor=LAB_COL["NG"], label="Negative (NG)"),
                   Patch(facecolor=LAB_COL["EXCLUDE"], label="excluded (below consensus gate)")]
    else:
        handles = [Patch(facecolor=SPK_COL["child"], label="child"),
                   Patch(facecolor=SPK_COL["therapist"], label="therapist"),
                   Patch(facecolor=GREY, label="excluded (below consensus gate)")]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8,
               frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"{out}.{ext}"), dpi=200, bbox_inches="tight")
    plt.close(fig)

draw("emotion", "session_figure")
draw("speaker", "speaker_figure")
print("wrote session_figure.pdf/.png and speaker_figure.pdf/.png")
display(Image(os.path.join(OUT_DIR, "session_figure.png")))
display(Image(os.path.join(OUT_DIR, "speaker_figure.png")))
